# Zero init, step by step: how layer 2 unlocks layer 1

`train_with_zero_initialization.py` starts **both** $W_{qk}$ matrices at exactly zero. The companion
notebook `zero_init_analysis.ipynb` looks at this run at **epoch** resolution (1000 epochs, one snapshot
per epoch). This notebook zooms into the part that resolution cannot see: the **first ~20 optimizer
steps**, where one epoch is 98 steps.

The finding, stated up front:

> At the zero init, $\partial L/\partial W^1_{qk}$ is **exactly zero** — every one of its 11,664 entries —
> while $\partial L/\partial W^2_{qk}$ is not. Layer 1 cannot move on step 1. Layer 2 moves, and *that*
> is what makes layer 1's gradient nonzero from step 2 onward. Layer 1's gradient is **proportional
> to $W^2_{qk}$**.

This is the zero-init counterpart of the observation in `meeting_notes.txt` (2nd theorem): *"the $W_{qk}$
for the second layer has the right gradient signal even at the random init, and the qk matrix of the
first layer has no meaningful gradient signal — the second layer forms first."* At random init that is
a statement about relative magnitudes. At zero init it becomes exact: not *weak*, but *identically zero*,
and provably so.

**Contents**

1. The first gradient — the claim, the proof, and the picture
2. Test A — pin $W^2_{qk}$ at zero and layer 1 never moves
3. Test B — dose–response: $\|\partial L/\partial W^1_{qk}\| \propto \|W^2_{qk}\|$
4. Verifying the two algebraic claims directly
5. The first 20 steps
6. Animations
7. Where does the very first gradient land?
8. Takeaways

Everything runs on CPU in well under a minute (~0.12 s per step at the real batch size of 256).

## 0. Setup

Constants match the training script: $V=8$ ($V_{sig}=4$ signal + 4 noise), $T=100$, so
$D = V+T = 108$ (layer-1 width) and $\text{IN2} = 2(V+T) = 216$ (layer-2 width).

In [ ]:
import os, json, math, importlib.util
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.colors import SymLogNorm
from IPython.display import HTML, display

torch.set_num_threads(8)

V, VS, T = 8, 4, 100
D, IN2   = V + T, 2 * (V + T)
SD       = D ** 0.5
ALPHA_REC, GAMMA, P1, P2, NF, BIG = 20.0, -50000.0, 1000.0, 50.0, 10.0, -1e9

THR    = 0.6          # primary threshold for the step-level study
SEED   = 0            # same seed the training script uses
B      = 256          # the real training batch size
STEPS  = 20           # how many optimizer steps to trace
DEVICE = 'cpu'        # 20 steps is ~2.5 s here; the box's GPUs are usually saturated

# ---- the training script, imported as a module (same trick as zero_init_analysis.ipynb) ----
_s = importlib.util.spec_from_file_location('ZI', 'train_with_zero_initialization.py')
ZI = importlib.util.module_from_spec(_s); _s.loader.exec_module(ZI)

TS, TE = ZI.CFG['token_start'], ZI.CFG['seq_len']       # scored window: tokens 50:100
LR     = ZI.CFG['lr']
assert (V, VS, T) == (ZI.CFG['v_sig'] + ZI.CFG['n_noise'], ZI.CFG['v_sig'], ZI.CFG['seq_len'])

plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 140, 'font.size': 11, 'axes.grid': False,
                     'animation.embed_limit': 200})
C1, C2 = '#d62728', '#1f77b4'          # layer 1 / layer 2 throughout
FIGDIR = 'artifacts/figures_zero_step'
os.makedirs(FIGDIR, exist_ok=True)

# ---- the construction, only used in section 7 as a reference support ----------------------
Wq1 = torch.eye(D) * SD
T1  = torch.zeros(D, D)
for k in range(VS, V):
    T1[k, :] = BIG
for tq in range(T):
    for tk in range(T):
        T1[V + tk, V + tq] = BIG if tq == tk else -ALPHA_REC * (tq - tk)
Wq2 = torch.zeros(D, IN2)
Wq2[:VS, :VS]      = torch.eye(VS) * SD * P1
Wq2[:VS, D:D + VS] = torch.eye(VS) * SD * P2
for v in range(V):
    Wq2[VS, v] = SD * NF
Wk2 = torch.zeros(D, IN2)
Wk2[:V, D:D + V] = torch.eye(V)
for v in range(VS, V):
    Wk2[VS, v] = GAMMA
CQK  = [Wq1.t() @ T1.t(), Wq2.t() @ Wk2]
MASK = [np.zeros((D, D), bool), np.zeros((IN2, IN2), bool)]
MASK[0][:, VS:V] = True; MASK[0][V:D, V:D] = True
MASK[1][0:V, 0:V] = True; MASK[1][0:V, D:D + V] = True; MASK[1][D:D + V, D:D + V] = True

# ---- data ---------------------------------------------------------------------------------
blob = ZI.load_data('train', THR)
X, Y = blob['x'], blob['y']
print(f'train {tuple(X.shape)}  alpha={THR}  succ_split={blob["succ_split"]}  '
      f'scored window {TS}:{TE}  batch={B}  lr={LR}')

# ---- helpers --------------------------------------------------------------------------------
def bounds(n):
    return [V] if n == D else ([V, D, D + V] if n == IN2 else [])

def show(ax, M, title, vmax=None, lw=0.8, cmap='RdBu_r'):
    '''symlog heatmap with the block boundaries drawn in; safe on an all-zero matrix.'''
    A = np.asarray(M.detach().cpu() if torch.is_tensor(M) else M, dtype=np.float64)
    v = float(vmax) if vmax else float(np.abs(A).max())
    if v <= 0:
        im = ax.imshow(A, cmap=cmap, interpolation='nearest', vmin=-1, vmax=1)
    else:
        im = ax.imshow(A, cmap=cmap, interpolation='nearest',
                       norm=SymLogNorm(linthresh=v / 1e3, linscale=0.6, vmin=-v, vmax=v, base=10))
    for b in bounds(A.shape[0]): ax.axhline(b - .5, color='k', lw=lw)
    for b in bounds(A.shape[1]): ax.axvline(b - .5, color='k', lw=lw)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f'{title}\n|max| {np.abs(A).max():.4g}   $\\|\\cdot\\|_F$ {np.linalg.norm(A):.4g}',
                 fontsize=10)
    return im

def annot_block(ax, A, title, vmax=None, fs=7.5):
    A = np.asarray(A, dtype=np.float64)
    v = float(vmax) if vmax else float(np.abs(A).max())
    if v <= 0:
        ax.imshow(A, cmap='RdBu_r', interpolation='nearest', vmin=-1, vmax=1)
    else:
        ax.imshow(A, cmap='RdBu_r', interpolation='nearest',
                  norm=SymLogNorm(linthresh=v / 1e3, linscale=0.6, vmin=-v, vmax=v, base=10))
    for i in range(A.shape[0]):
        for j in range(A.shape[1]):
            ax.text(j, i, '.' if A[i, j] == 0 else f'{A[i, j]:.3g}',
                    ha='center', va='center', fontsize=fs)
    ax.set_xticks(range(A.shape[1])); ax.set_yticks(range(A.shape[0]))
    ax.set_xticklabels([f'k{v_}' for v_ in range(A.shape[1])], fontsize=8)
    ax.set_yticklabels([f'q{v_}' for v_ in range(A.shape[0])], fontsize=8)
    ax.set_title(title, fontsize=11)

def share(A, mask):
    A = np.asarray(A, dtype=np.float64); tot = float((A ** 2).sum())
    return float((A[mask] ** 2).sum() / tot) if tot > 0 else float('nan')

def batch(k):
    '''batch k (0-indexed), wrapping — deterministic so every experiment below sees the same data.'''
    i = (k * B) % (X.shape[0] - B)
    return X[i:i + B].to(DEVICE), Y[i:i + B, TS:TE].to(DEVICE)

## 1. The first gradient

### What is being trained

`install_fixed` pins everything except the two $W_{qk}$ matrices:

* $W_{ov}^{(1)} = W_{ov}^{(2)} = \begin{bmatrix} I_V & 0\\ 0 & 0\end{bmatrix}$ — the value path copies the
  **raw token one-hot** and nothing else;
* `fc_out` is the identity selector onto layer 2's attention output;
* readout is $\log(f + b)$ with the bias applied **before** the log.

So the only free parameters are $W^1_{qk}\in\mathbb R^{108\times108}$ and $W^2_{qk}\in\mathbb R^{216\times216}$,
and `apply_init` sets both to exactly zero.

### Notation (matching `CombinedAttentionBlock.forward`)

$$X^0 = [\,\text{tok}\mid\text{pos}\,],\qquad
S^\ell = \tfrac{1}{\sqrt m}X^{\ell-1}W^\ell X^{\ell-1\top} + M,\qquad
A^\ell = \mathrm{softmax}(S^\ell),\qquad
Z^\ell = A^\ell (X^{\ell-1}V^\ell)$$

$$X^1 = [\,X^0\mid Z^1\,]\in\mathbb R^{T\times216},\qquad
\text{logits} = \log\!\big(Z^2_{[:,:V]} + b\big)$$

### Why $W_{qk}=0$ is not a degenerate state

$W_{qk}=0$ gives $S=M$, hence **uniform attention over the causal prefix**, not zero attention:

$$A_{ij} = \tfrac{1}{i+1}\ (j\le i)\qquad\Longrightarrow\qquad
Z^2_{i,[:V]} = \tfrac{1}{i+1}\sum_{j\le i}\text{tok}_j \;=\; f_i$$

$f_i$ is the **empirical token frequency of the prefix**. At the zero init the model predicts the
prefix histogram — a real, non-trivial predictor. The cell below checks this to $10^{-8}$.

### Claim A — layer 1's gradient vanishes identically

Since $V^2$ is nonzero only in rows $0..V\!-\!1$ and $X^1_{[:,0:V]}=\text{tok}$,

$$X^1V^2 = [\,\text{tok}\mid 0\,] =: P$$

$P$ contains **no $W^1$**: the columns of $X^1$ carrying $Z^1$ multiply rows $108..215$ of $V^2$, which
are zero. So layer 1 reaches the loss only through $A^2$. Block-partitioning $W^2$ by the two halves
of $X^1$ ($a\leftrightarrow X^0$, $b\leftrightarrow Z^1$),

$$\frac{\partial(\sqrt m\,S^2_{ij})}{\partial Z^1_k}
= \delta_{ik}\big(W_{ba}X^0_j + W_{bb}Z^1_j\big) + \delta_{jk}\big(W_{ab}^\top X^0_i + W_{bb}^\top Z^1_i\big)$$

Every term carries **exactly one factor of a block of $W^2$**, so

$$W^2 = 0 \;\Longrightarrow\; \frac{\partial S^2}{\partial Z^1} = 0
\;\Longrightarrow\; \frac{\partial L}{\partial Z^1} = 0
\;\Longrightarrow\; \boxed{\ \frac{\partial L}{\partial W^1} = 0\ }$$

regardless of how large $\partial L/\partial S^2$ is. This is **structural**: it holds for every batch and
every dataset, exactly.

### Claim B — layer 2's gradient does not vanish

$$\frac{\partial L}{\partial W^2}=\frac{1}{\sqrt m}\sum_b\sum_{i,j} c^{(b)}_{ij}\;X^1_i X^{1\top}_j,
\qquad c_{ij}:=\frac{\partial L}{\partial S^2_{ij}}$$

No prefactor of $W^2$. $W^1$ enters only through $X^1$, and $W^1=0$ leaves $Z^1\neq0$ (uniform attention),
so nothing is killed. Because $\text{pos}_i$ is a **one-hot at coordinate $V+i$** of $X^1$, the
position–position block reads off exactly, with no cancellation:

$$\boxed{\ \left[\frac{\partial L}{\partial W^2}\right]_{V+i,\,V+j} = \frac{1}{\sqrt m}\sum_b c^{(b)}_{ij}\ }$$

And $c_{ij}\neq0$: writing $g_{i,v} := \dfrac{p_{i,v}-\mathbb 1[v=y_i]}{N\,(f_{i,v}+b_v)}$, we get
$\partial L/\partial A^2_{ij} = g_{i,x_j}$, and through the softmax Jacobian

$$c_{ij}=\frac{1}{i+1}\big(g_{i,x_j}-\bar g_i\big),\qquad \bar g_i=\tfrac{1}{i+1}\textstyle\sum_{k\le i}g_{i,x_k}$$

Since $g_{i,y_i}<0<g_{i,v}$ for $v\neq y_i$ (strict), $c_{i\cdot}\equiv0$ would require the whole prefix
to be either all $y_i$ or to contain no $y_i$. For a scored position $i\ge50$ over 8 symbols that fails.

**A caveat worth stating in any write-up.** Claim A is an identity — multiplication by zero, so it holds
always. Claim B is *generic*: a batch sums $\sum_b c^{(b)}_{ij}$, so vanishing would need exact
cancellation across 256 sequences simultaneously in all $T^2$ entries. Not impossible, just a
measure-zero coincidence. Layer 1 **cannot** be nonzero; layer 2 **won't** be zero.

In [ ]:
def build(w2_scale=0.0, w2_seed=7):
    '''A model exactly as train_one() builds it: zero W_qk, W_ov + unembedding pinned.
    w2_scale > 0 overrides W_qk2 with a random matrix of that scale (used in section 3).'''
    mod = ZI.QKOnlyTransformer(V, T, D, VS, THR).to(DEVICE)
    ZI.apply_init(mod, SEED)                     # <- W_qk1 = W_qk2 = 0
    params = ZI.install_fixed(mod)               # <- freeze all, re-enable the two W_qk
    if w2_scale:
        torch.manual_seed(w2_seed)
        with torch.no_grad():
            mod.layer2.W_qk.copy_(torch.randn_like(mod.layer2.W_qk) * w2_scale)
    return mod, params

def grad_step(mod, k=0, do_step=None):
    '''forward/backward on batch k; returns (loss, grad Wqk1, grad Wqk2) as detached copies.'''
    xb, yb = batch(k)
    for p in (mod.layer1.W_qk, mod.layer2.W_qk):
        p.grad = None
    loss = F.cross_entropy(mod(xb)[:, TS:TE].reshape(-1, V), yb.reshape(-1))
    loss.backward()
    g1 = mod.layer1.W_qk.grad.detach().clone()
    g2 = mod.layer2.W_qk.grad.detach().clone()
    if do_step is not None:
        do_step.step()
    return float(loss), g1, g2

mod, params = build()
print('trainable:', [tuple(p.shape) for p in params],
      f'= {sum(p.numel() for p in params):,} params')
print(f'|W_qk1| = {mod.layer1.W_qk.norm():.1f}   |W_qk2| = {mod.layer2.W_qk.norm():.1f}   '
      f'(both exactly zero, as apply_init intends)')

# --- the zero init predicts the prefix histogram: Z2[:, :V] == running token frequency ---
with torch.no_grad():
    xb, _ = batch(0)
    s = mod.stream(xb)
    Z2 = s[..., mod.in2:mod.in2 + V]
    tok = F.one_hot(xb, num_classes=V).float()
    f = torch.cumsum(tok, 1) / torch.arange(1, T + 1, device=DEVICE).view(1, -1, 1)
print(f'max |Z2[:, :V] - prefix token frequency| = {(Z2 - f).abs().max():.3e}   '
      '(so logits = log(prefix histogram + b) at step 0)')

### The picture: gradients at step 1

Left column: the two gradients at the zero init. Layer 1's is not merely small — `np.count_nonzero`
returns 0 of 11,664.

In [ ]:
mod, params = build()
loss0, G1_0, G2_0 = grad_step(mod, k=0)

print(f'loss at the zero init : {loss0:.6f}   (ln 8 = {math.log(8):.4f} for a uniform predictor)')
for nm, g in (('grad W_qk layer 1', G1_0), ('grad W_qk layer 2', G2_0)):
    a = g.abs()
    print(f'{nm}: shape={tuple(g.shape)}  ||.||_F={g.norm():.6e}  |max|={a.max():.6e}  '
          f'nonzero={int((g != 0).sum()):,}/{g.numel():,}')

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
show(axes[0], G1_0, r'$\partial L/\partial W^1_{qk}$  at the zero init'
               '\n(identically zero — flat panel is the result)')
show(axes[1], G2_0, r'$\partial L/\partial W^2_{qk}$  at the zero init')
fig.suptitle(f'Step-1 gradients from $W^1_{{qk}}=W^2_{{qk}}=0$   ($\\alpha$={THR}, batch={B})',
             fontsize=14)
fig.tight_layout()
fig.savefig(f'{FIGDIR}/step1_gradients_a{THR}.png', bbox_inches='tight')
plt.show()

### 1b. Why the weights do not look like the negative of the gradient

A natural expectation from the heatmaps above: after one step the weights should be $-\text{lr}\cdot g$,
so the $W^2_{qk}$ picture should be the gradient picture with the colours flipped. It is not, and the
reason is **Adam**, not the gradient.

Starting from $W=0$ with no optimizer state, the first update is

$$m_1=(1-\beta_1)g,\quad v_1=(1-\beta_2)g^2
\;\Longrightarrow\;
\hat m=\frac{m_1}{1-\beta_1}=g,\quad \hat v=\frac{v_1}{1-\beta_2}=g^2$$

$$\Delta W=-\,\text{lr}\cdot\frac{\hat m}{\sqrt{\hat v}+\epsilon}
=-\,\text{lr}\cdot\frac{g}{|g|+\epsilon}\;\approx\;-\,\text{lr}\cdot\mathrm{sign}(g)$$

The second moment divides each coordinate by **its own** gradient magnitude, so every coordinate takes
the same size step $\pm\text{lr}$ whether its gradient was $10^{-4}$ or $10^{-9}$. The *sign* pattern
survives perfectly; the *magnitude* pattern is destroyed. With plain SGD the two heatmaps would agree
up to sign and a scale factor.

This is also the reason `induction_diag_off` is exactly 1.0 at step 1 in section 6b: the diagonal's
dominance lives entirely in the gradient magnitudes, which is precisely what Adam discards on step 1.

After step 1 the relationship is looser still — momentum mixes in past gradients and every step draws a
different batch, so signs flip cell to cell. The practical way to read Animation 1: **a cell whose
magnitude grows steadily is one whose gradient sign is consistent across batches.**

In [ ]:
mod_a, params_a = build()
opt_a1 = torch.optim.Adam(params_a, lr=LR, weight_decay=ZI.CFG['weight_decay'])
_, _, G2a = grad_step(mod_a, k=0, do_step=opt_a1)
W2a = mod_a.layer2.W_qk.detach().clone()
eps = opt_a1.param_groups[0]['eps']

nz = G2a != 0
print(f'gradient: nonzero {int(nz.sum()):,}/{G2a.numel():,}   '
      f'|g| in [{G2a[nz].abs().min():.3e}, {G2a.abs().max():.3e}]')
print(f'weights : nonzero {int((W2a != 0).sum()):,}   '
      f'|w| in [{W2a[W2a != 0].abs().min():.3e}, {W2a.abs().max():.3e}]   lr = {LR}')
print(f'  -> the support is preserved exactly; the magnitudes collapse onto lr.\n')
print(f'max |W - (-lr * sign(g))|         = {(W2a + LR*torch.sign(G2a)).abs().max():.3e}')
print(f'max |W - (-lr * g/(|g|+eps))|     = '
      f'{(W2a + LR*G2a/(G2a.abs()+eps)).abs().max():.3e}   <- the exact identity')
print(f'\ncosine(W after step 1, -gradient)  = '
      f'{float((W2a[nz]@(-G2a[nz]))/(W2a[nz].norm()*G2a[nz].norm())):.4f}   '
      '(1.0000 would mean "proportional", i.e. plain SGD)')
print(f'cosine(sign W, sign -gradient)    = '
      f'{float(torch.sign(W2a[nz])@torch.sign(-G2a[nz])/nz.sum()):.4f}   '
      '(the sign pattern IS carried over, perfectly)')

fig, axes = plt.subplots(1, 2, figsize=(13.5, 6.4))
annot_blk = lambda ax, A, t, v: annot_block(ax, A, t, vmax=v)
annot_blk(axes[0], G2a[:V, D:D + V].numpy() * 1e5,
          r'$\partial L/\partial W^2_{qk}[0{:}8,\ 108{:}116]$   ($\times 10^{-5}$)  — graded', None)
annot_blk(axes[1], W2a[:V, D:D + V].numpy() * 1e3,
          r'$W^2_{qk}$ after one Adam step   ($\times 10^{-3}$)  — flat $\pm$lr', None)
fig.suptitle('Adam keeps the sign of the gradient and discards its magnitude', fontsize=13)
fig.tight_layout(); fig.savefig(f'{FIGDIR}/adam_sign_only_a{THR}.png', bbox_inches='tight')
plt.show()

Claim A says this is exact and dataset-independent, so it should hold at every $\alpha$.

In [ ]:
print(f'{"alpha":>6s} {"loss":>10s} {"||g Wqk1||":>13s} {"nonzero g1":>11s} {"||g Wqk2||":>13s} '
      f'{"nonzero g2":>11s}')
_thr_keep = THR
for t in (0.6, 0.8, 1.0):
    THR = t
    blob_t = ZI.load_data('train', t); X, Y = blob_t['x'], blob_t['y']
    m_t, _ = build()
    l_t, g1_t, g2_t = grad_step(m_t, k=0)
    print(f'{t:6.1f} {l_t:10.5f} {g1_t.norm():13.4e} {int((g1_t != 0).sum()):11d} '
          f'{g2_t.norm():13.4e} {int((g2_t != 0).sum()):11d}')
THR = _thr_keep
blob = ZI.load_data('train', THR); X, Y = blob['x'], blob['y']
print(f'\nrestored alpha={THR}')

## 2. Test A — pin $W^2_{qk}$ at zero

Claim A says layer 1 is not *delayed*, it is **gated**. If $W^2$ is held at zero forever, layer 1's
gradient must stay exactly zero forever — no amount of training can start it.

Below, only $W^1_{qk}$ is handed to the optimizer, so $W^2_{qk}$ never updates.

In [ ]:
mod, _ = build()
opt_a = torch.optim.Adam([mod.layer1.W_qk], lr=LR)      # W_qk2 deliberately excluded

rows = []
for k in range(8):
    l, g1, g2 = grad_step(mod, k=k, do_step=opt_a)
    rows.append((k + 1, l, float(g1.norm()), float(mod.layer1.W_qk.norm()),
                 float(mod.layer2.W_qk.norm())))

print('TEST A -- W_qk2 pinned at zero, only W_qk1 in the optimizer')
print(f'{"step":>5} {"loss":>10} {"||g Wqk1||":>13} {"||Wqk1||":>11} {"||Wqk2||":>11}')
for r in rows:
    print(f'{r[0]:>5} {r[1]:>10.5f} {r[2]:>13.6e} {r[3]:>11.4f} {r[4]:>11.4f}')

ok = all(r[2] == 0.0 and r[3] == 0.0 for r in rows)
print(f'\nlayer 1 gradient identically zero at every step: {ok}   '
      '-> layer 1 is gated by W_qk2, not merely slow')

## 3. Test B — dose–response

Claim A also predicts the *functional form*. $\partial S^2/\partial Z^1$ is exactly **linear** in $W^2$, so
for small $W^2$

$$\frac{\partial L}{\partial W^1} = C\cdot W^2 + O(\|W^2\|^2)$$

Setting $W^2$ by hand **before** step 1 (with $W^1$ still zero) should therefore give a gradient that
scales one-for-one with $\|W^2\|$ — a slope of exactly 1 on log–log axes.

In [ ]:
scales = [0.0, 1e-4, 1e-3, 1e-2, 1e-1, 1.0]
n2, n1 = [], []
print('TEST B -- W_qk2 set nonzero BEFORE step 1, W_qk1 still zero')
print(f'{"||Wqk2||":>12} {"||g Wqk1||":>15} {"ratio to previous":>19}')
prev = None
for s in scales:
    m_b, _ = build(w2_scale=s)
    _, g1_b, _ = grad_step(m_b, k=0)
    a, b_ = float(m_b.layer2.W_qk.norm()), float(g1_b.norm())
    n2.append(a); n1.append(b_)
    r = '-' if not prev else f'{b_ / prev:.3f}x'
    print(f'{a:12.4f} {b_:15.6e} {r:>19}')
    prev = b_ if b_ > 0 else None

xs, ys = np.array(n2[1:]), np.array(n1[1:])
slope = np.polyfit(np.log10(xs), np.log10(ys), 1)[0]

fig, ax = plt.subplots(figsize=(7.2, 5.4))
ax.loglog(xs, ys, 'o-', color='#d62728', lw=1.6, ms=7, label='measured')
ax.loglog(xs, ys[0] * xs / xs[0], '--', color='k', lw=1.1, label='slope 1 (theory)')
ax.set_xlabel(r'$\|W^2_{qk}\|_F$ imposed before step 1')
ax.set_ylabel(r'$\|\partial L/\partial W^1_{qk}\|_F$ at step 1')
ax.set_title(f'Layer 1\'s gradient is proportional to $W^2_{{qk}}$   (fitted slope {slope:.4f})')
ax.grid(True, which='both', alpha=0.3); ax.legend()
fig.tight_layout(); fig.savefig(f'{FIGDIR}/dose_response_a{THR}.png', bbox_inches='tight')
plt.show()
print(f'\nfitted log-log slope = {slope:.4f}  (1.0 = exact proportionality)')
print('the largest scale bends upward slightly: at ||W2|| ~ 200 the softmax saturates and the')
print('first-order term is no longer the whole story. The training run never goes there early on.')

## 4. Verifying the two algebraic claims directly

Two checks that go past "the numbers came out zero" and test the actual derivation.

**Check 1 (Claim A).** Split $\partial L/\partial X^1$ into its two blocks: coordinates $0..107$ ($X^0$) and
$108..215$ ($Z^1$). The proof says the **$Z^1$ block is exactly zero** when $W^2=0$, while the $X^0$ block
stays nonzero (the value path carries raw tokens to the readout regardless).

**Check 2 (Claim B).** The position–position block identity
$\left[\partial L/\partial W^2\right]_{V+i,V+j} = \frac{1}{\sqrt m}\sum_b c^{(b)}_{ij}$ should hold to
floating-point precision.

In [ ]:
def dissect(w2_scale=0.0, Bn=B):
    '''manual forward replicating stream(), exposing X1 and S2 so their grads can be read.'''
    mod, _ = build(w2_scale=w2_scale)
    m_out = mod.layer2.output_dim
    xb, yb = X[:Bn].to(DEVICE), Y[:Bn, TS:TE].to(DEVICE)
    tok = F.one_hot(xb, num_classes=V).float()
    pos = F.one_hot(torch.arange(T, device=DEVICE), num_classes=T).float().unsqueeze(0).expand(Bn, T, -1)
    X0   = torch.cat([tok, pos], -1)
    mask = torch.triu(torch.ones(T, T, device=DEVICE) * float('-inf'), diagonal=1)
    X1 = mod.layer1(X0, mask); X1.retain_grad()
    S2 = torch.bmm(X1 @ mod.layer2.W_qk, X1.transpose(1, 2)) / (m_out ** 0.5) + mask
    S2.retain_grad()
    Z2 = torch.bmm(F.softmax(S2, -1), X1 @ mod.layer2.W_ov)
    X2 = torch.cat([X1, Z2], -1)
    sl = slice(mod.in2, mod.in2 + V)
    z  = X2.clone()
    z[..., sl] = torch.log((X2[..., sl] + mod.readout_bias).clamp_min(ZI.CFG['log_floor']))
    loss = F.cross_entropy(mod.fc_out(z)[:, TS:TE].reshape(-1, V), yb.reshape(-1))
    loss.backward()
    return mod, X1, S2, m_out

print('CHECK 1 -- dL/dX1, split into the X0 block and the Z1 block')
print(f'{"W2 scale":>10} {"max|dL/dX0|":>15} {"max|dL/dZ1|":>15} {"max|dL/dWqk1|":>16}')
for s in (0.0, 1e-3, 1e-2, 1e-1):
    mod_c, X1_c, _, _ = dissect(s, Bn=64)
    print(f'{s:>10} {X1_c.grad[..., :D].abs().max():15.6e} '
          f'{X1_c.grad[..., D:].abs().max():15.6e} '
          f'{mod_c.layer1.W_qk.grad.abs().max():16.6e}')
print('  -> the Z1 block is exactly 0 at W2=0 and scales linearly after; the X0 block never dies.\n')

print('CHECK 2 -- position-block identity  [dL/dW2]_{V+i,V+j} == (1/sqrt(m)) sum_b dL/dS2_{b,i,j}')
print(f'{"batch":>7} {"max|LHS - RHS|":>17} {"max|LHS|":>14}')
for Bn in (1, 4, B):
    mod_c, _, S2_c, m_out = dissect(0.0, Bn=Bn)
    lhs = mod_c.layer2.W_qk.grad[V:V + T, V:V + T]
    rhs = S2_c.grad.nan_to_num().sum(0) / (m_out ** 0.5)
    print(f'{Bn:>7} {(lhs - rhs).abs().max():17.4e} {lhs.abs().max():14.6e}')
print('  -> exact to float precision. So dL/dW2 = 0 would force dL/dS2 = 0, which section 1 rules out.')

## 5. The first 20 steps

Now run the real thing — Adam, `lr=0.005`, batch 256, both matrices trainable, exactly as `train_one`
does — and record $W_{qk}$ and its gradient at every step.

Bookkeeping: `Wsnap[k]` are the weights **before** step $k+1$ (so `Wsnap[0]` is the zero init), and
`Gsnap[k]` is the gradient evaluated at `Wsnap[k]`. Hence `Gsnap[0]` is the step-1 gradient from section 1.

In [ ]:
mod, params = build()
opt = torch.optim.Adam(params, lr=LR, weight_decay=ZI.CFG['weight_decay'])

Wsnap = [[], []]       # [layer][step] weights BEFORE the update
Gsnap = [[], []]       # [layer][step] gradient AT those weights
LOSS  = []

for k in range(STEPS):
    Wsnap[0].append(mod.layer1.W_qk.detach().cpu().numpy().copy())
    Wsnap[1].append(mod.layer2.W_qk.detach().cpu().numpy().copy())
    l, g1, g2 = grad_step(mod, k=k, do_step=opt)
    Gsnap[0].append(g1.cpu().numpy()); Gsnap[1].append(g2.cpu().numpy())
    LOSS.append(l)
Wsnap[0].append(mod.layer1.W_qk.detach().cpu().numpy().copy())
Wsnap[1].append(mod.layer2.W_qk.detach().cpu().numpy().copy())
Wsnap = [np.asarray(a) for a in Wsnap]; Gsnap = [np.asarray(a) for a in Gsnap]

gn  = [np.linalg.norm(Gsnap[i].reshape(STEPS, -1), axis=1) for i in (0, 1)]
wn  = [np.linalg.norm(Wsnap[i].reshape(STEPS + 1, -1), axis=1) for i in (0, 1)]
st  = np.arange(1, STEPS + 1)

print(f'{"step":>5} {"loss":>10} {"||g Wqk1||":>13} {"||g Wqk2||":>13} {"g1/g2":>9} '
      f'{"||Wqk1||":>10} {"||Wqk2||":>10} {"W1/W2":>8}')
for k in range(STEPS):
    print(f'{k+1:>5} {LOSS[k]:>10.5f} {gn[0][k]:>13.4e} {gn[1][k]:>13.4e} '
          f'{gn[0][k]/gn[1][k]:>9.4f} {wn[0][k+1]:>10.4f} {wn[1][k+1]:>10.4f} '
          f'{wn[0][k+1]/wn[1][k+1]:>8.4f}')
print(f'\none epoch = {X.shape[0]}/{B} = {math.ceil(X.shape[0]/B)} steps, so all of this is inside '
      f'epoch 1 -- invisible to the per-epoch snapshots in zero_init_analysis.ipynb.')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5.0))

pos = gn[0] > 0                      # step 1 is exactly 0 and has no place on a log axis
lo, hi = gn[0][pos].min() / 6, gn[1].max() * 4
axes[0].semilogy(st[pos], gn[0][pos], 'o-', color=C1, lw=1.5, ms=4, label='layer 1')
axes[0].semilogy(st, gn[1], 's-', color=C2, lw=1.5, ms=4, label='layer 2')
axes[0].set_ylim(lo, hi)
axes[0].axvspan(0.5, 1.5, color='k', alpha=0.09)
axes[0].plot([1], [lo * 1.5], marker='v', color=C1, ms=12, clip_on=False)
axes[0].annotate('step 1: layer 1 gradient is\nexactly 0 (no log position)',
                 xy=(1, lo * 1.5), xytext=(3.4, lo * 14), fontsize=9,
                 arrowprops=dict(arrowstyle='->', lw=0.9))
axes[0].set_title(r'gradient norm  $\|\partial L/\partial W_{qk}\|_F$')
axes[0].set_xlabel('optimizer step'); axes[0].legend(loc='center right')

axes[1].plot(np.arange(STEPS + 1), wn[0], 'o-', color=C1, lw=1.5, ms=4, label='layer 1')
axes[1].plot(np.arange(STEPS + 1), wn[1], 's-', color=C2, lw=1.5, ms=4, label='layer 2')
axes[1].axvspan(0, 1, color='k', alpha=0.09)
axes[1].set_title(r'weight norm  $\|W_{qk}\|_F$   (layer 1 flat over step 1)')
axes[1].set_xlabel('optimizer step'); axes[1].legend()

axes[2].plot(st, gn[0] / gn[1], 'o-', color='#2ca02c', lw=1.5, ms=4, label=r'$\|g_1\|/\|g_2\|$')
axes[2].plot(np.arange(STEPS + 1), wn[0] / np.maximum(wn[1], 1e-30), 's-', color='#9467bd',
             lw=1.5, ms=4, label=r'$\|W_1\|/\|W_2\|$')
axes[2].axhline(1.0, color='k', ls='--', lw=0.9)
axes[2].set_title('layer 1 relative to layer 2\n(both start at 0, neither has caught up by step 20)')
axes[2].set_xlabel('optimizer step'); axes[2].legend()

for a in axes:
    a.grid(alpha=0.3)
fig.suptitle(f'First {STEPS} optimizer steps from the zero init   '
             f'($\\alpha$={THR}, Adam lr={LR}, batch={B})', fontsize=14)
fig.tight_layout()
fig.savefig(f'{FIGDIR}/first{STEPS}_steps_a{THR}.png', bbox_inches='tight')
plt.show()

## 6. Animations

**Animation 1** — all four quantities at once. Frame $k$ shows $W_{qk}$ **before** step $k{+}1$ and the
gradient computed at those weights. Watch frame 0: the layer-1 gradient panel is blank while the
layer-2 panel already carries structure. From frame 1 onward the layer-1 panels come alive.

Colour scales are **fixed across frames** (per panel, symlog), so growth is real and not rescaling.

In [ ]:
def four_panel_animation(Wsnap, Gsnap, steps, fname, interval=520):
    vmax = [float(np.abs(Wsnap[0]).max()), float(np.abs(Wsnap[1]).max()),
            float(np.abs(Gsnap[0]).max()), float(np.abs(Gsnap[1]).max())]
    fig, axes = plt.subplots(2, 2, figsize=(11.5, 11.0))
    fig.subplots_adjust(top=0.90)

    def draw(k):
        for ax in axes.ravel():
            ax.clear()
        show(axes[0][0], Wsnap[0][k], r'$W^1_{qk}$   (108$\times$108)', vmax=vmax[0])
        show(axes[0][1], Wsnap[1][k], r'$W^2_{qk}$   (216$\times$216)', vmax=vmax[1])
        show(axes[1][0], Gsnap[0][k], r'$\partial L/\partial W^1_{qk}$', vmax=vmax[2])
        show(axes[1][1], Gsnap[1][k], r'$\partial L/\partial W^2_{qk}$', vmax=vmax[3])
        tag = '  --  layer 1 frozen: gradient identically zero' if k == 0 else ''
        fig.suptitle(f'step {k}  (weights before update {k+1}, gradient at those weights){tag}',
                     fontsize=13)
        return []

    anim = animation.FuncAnimation(fig, draw, frames=steps, interval=interval, blit=False)
    anim.save(fname, writer=animation.PillowWriter(fps=max(1, int(1000 / interval))))
    plt.close(fig)
    return anim

anim1 = four_panel_animation(Wsnap, Gsnap, STEPS, f'{FIGDIR}/evolution_wqk_grad_a{THR}.gif')
print(f'saved {FIGDIR}/evolution_wqk_grad_a{THR}.gif')
display(HTML(anim1.to_jshtml()))

**Animation 2** — the induction block, $W^2_{qk}[0{:}8,\;108{:}116]$ (own token $\times$ key predecessor),
every value printed. This is the $8\times8$ corner the construction fills with $10392.3\,I_4$ plus a
$103.9$ trigger in column 4. Left: the weights. Right: the gradient writing them.

If the layer-2 gradient is not just nonzero but *pointed at the right place*, it shows here first.

In [ ]:
R, Cc = slice(0, V), slice(D, D + V)
Wblk = Wsnap[1][:, R, Cc]
Gblk = Gsnap[1][:, R, Cc]
vW, vG = float(np.abs(Wblk).max()), float(np.abs(Gblk).max())

fig, axes = plt.subplots(1, 2, figsize=(13.5, 6.4))
fig.subplots_adjust(top=0.86)

def draw2(k):
    for ax in axes: ax.clear()
    annot_block(axes[0], Wblk[k], r'$W^2_{qk}[0{:}8,\ 108{:}116]$', vmax=vW)
    annot_block(axes[1], Gblk[k], r'$\partial L/\partial W^2_{qk}$  (same block)', vmax=vG)
    fig.suptitle(f'induction block, step {k}   '
                 f'(construction target: $10392.3\\,I_4$ + trigger 103.9 in column 4)', fontsize=12)
    return []

anim2 = animation.FuncAnimation(fig, draw2, frames=STEPS, interval=520, blit=False)
anim2.save(f'{FIGDIR}/evolution_induction_block_a{THR}.gif',
           writer=animation.PillowWriter(fps=2))
plt.close(fig)
print(f'saved {FIGDIR}/evolution_induction_block_a{THR}.gif')
display(HTML(anim2.to_jshtml()))

### Is the induction diagonal already forming?

The training script logs `induction_diag_off` every epoch — mean$|$diagonal$|$ over mean$|$off-diagonal$|$
of the $4\times4$ signal sub-block $W^2_{qk}[0{:}4,\,108{:}112]$ (`qk_stats`, line 346). A ratio of 1 means
"no diagonal structure"; the construction is a pure diagonal there.

Here is that same quantity at step resolution, plus the signed pressure GD is applying. Gradient descent
moves **against** the gradient, so the relevant direction is $-\partial L/\partial W^2_{qk}$: if its diagonal
entries are more positive than its off-diagonal entries, the diagonal is being actively built.

Two things to read carefully rather than over-read:

* The ratio is **exactly 1.0 at step 1** — not because the gradient is unstructured, but because Adam's
  first update is $\pm\text{lr}$ in every coordinate regardless of gradient magnitude ($m/\sqrt v$ with
  $v = m^2$ on the first step). Magnitude information only starts to register from step 2. The *right*
  panel is the honest view of step 1, and it already separates by an order of magnitude.
* The ratio is **non-monotonic** here: it peaks around step 7 and eases back. Twenty steps is a fifth of
  one epoch, so this is the very first stirring, not the trajectory. `zero_init_analysis.ipynb` §3 tracks
  the same metric over all 1000 epochs.

In [ ]:
def diag_off(blk4):
    a = np.abs(np.asarray(blk4, dtype=np.float64))
    dg = np.diag(a).mean()
    off = (a.sum() - np.diag(a).sum()) / (a.size - a.shape[0])
    return dg / off if off > 1e-30 else np.nan

W4 = Wsnap[1][:, :VS, D:D + VS]          # the 4x4 signal sub-block, same slice as qk_stats
G4 = Gsnap[1][:, :VS, D:D + VS]
eye = np.eye(VS, dtype=bool)

ratio   = np.array([diag_off(W4[k + 1]) for k in range(STEPS)])
push_dg = np.array([(-G4[k])[eye].mean() for k in range(STEPS)])
push_of = np.array([(-G4[k])[~eye].mean() for k in range(STEPS)])

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8))
axes[0].plot(st, ratio, 'o-', color=C2, lw=1.6, ms=4)
axes[0].axhline(1.0, color='k', ls='--', lw=1.0, label='no diagonal structure')
axes[0].set_xlabel('optimizer step'); axes[0].set_ylabel('induction_diag_off')
axes[0].set_title(r'diagonal structure in $W^2_{qk}[0{:}4,\ 108{:}112]$')
axes[0].grid(alpha=0.3); axes[0].legend()

axes[1].plot(st, push_dg, 'o-', color='#2ca02c', lw=1.6, ms=4, label='diagonal cells')
axes[1].plot(st, push_of, 's-', color='#ff7f0e', lw=1.6, ms=4, label='off-diagonal cells')
axes[1].axhline(0.0, color='k', ls='--', lw=1.0)
axes[1].set_xlabel('optimizer step')
axes[1].set_ylabel(r'mean of $-\partial L/\partial W^2_{qk}$')
axes[1].set_title('which cells GD is pushing up')
axes[1].grid(alpha=0.3); axes[1].legend()

fig.suptitle(f'The induction block over the first {STEPS} steps  ($\\alpha$={THR})', fontsize=13)
fig.tight_layout(); fig.savefig(f'{FIGDIR}/induction_diag_first_steps_a{THR}.png', bbox_inches='tight')
plt.show()

print(f'step 1  : diag push {push_dg[0]:+.3e}   off-diag push {push_of[0]:+.3e}   '
      f'ratio {push_dg[0]/push_of[0] if push_of[0] else float("nan"):.2f}x')
print(f'step {STEPS:<2}: diag push {push_dg[-1]:+.3e}   off-diag push {push_of[-1]:+.3e}   '
      f'induction_diag_off = {ratio[-1]:.3f}')

## 7. Where does the very first gradient land?

`zero_init_analysis.ipynb` §4 makes the point that the construction's layer-2 support is 192 cells out of
46,656 — **0.41%** — so a matrix spreading its mass uniformly would hold 0.41% there. At the **random**
init the *gradient* already sat at 0.48–0.89 in that support.

The zero init lets us ask the same question with no initial noise to inherit: of the very first gradient
GD ever computes, what fraction of its energy is inside the construction's support?

In [ ]:
base = [MASK[0].sum() / MASK[0].size, MASK[1].sum() / MASK[1].size]
print(f'construction support: layer 1 {MASK[0].sum():,}/{MASK[0].size:,} ({base[0]:.2%})   '
      f'layer 2 {MASK[1].sum():,}/{MASK[1].size:,} ({base[1]:.2%})\n')
print(f'{"step":>5} {"share(g2 in support)":>21} {"share(W2 in support)":>21} {"vs uniform":>12}')
sh_g, sh_w = [], []
for k in range(STEPS):
    a, b_ = share(Gsnap[1][k], MASK[1]), share(Wsnap[1][k + 1], MASK[1])
    sh_g.append(a); sh_w.append(b_)
    print(f'{k+1:>5} {a:>21.4f} {b_:>21.4f} {a/base[1]:>11.1f}x')

fig, ax = plt.subplots(figsize=(8.4, 5.2))
ax.plot(st, sh_g, 'o-', color=C2, lw=1.6, ms=4, label=r'gradient $\partial L/\partial W^2_{qk}$')
ax.plot(st, sh_w, 's-', color='#9467bd', lw=1.6, ms=4, label=r'weights $W^2_{qk}$')
ax.axhline(base[1], color='k', ls='--', lw=1.0,
           label=f'uniform baseline ({base[1]:.2%})')
ax.set_xlabel('optimizer step'); ax.set_ylabel('fraction of squared mass in the construction support')
ax.set_title('Layer 2: where the first gradients put their energy')
ax.grid(alpha=0.3); ax.legend()
fig.tight_layout(); fig.savefig(f'{FIGDIR}/support_share_first_steps_a{THR}.png', bbox_inches='tight')
plt.show()

## 8. Takeaways

1. **The zero init is not degenerate.** $W_{qk}=0$ means *uniform* attention, not *no* attention, so the
   model starts as the prefix-histogram predictor and the value path is fully alive (section 1).

2. **Layer 1 is gated, not slow.** $\partial L/\partial W^1_{qk} = 0$ exactly at step 1 — all 11,664 entries,
   at every $\alpha$ — because $\partial S^2/\partial Z^1$ carries $W^2$ as a multiplicative factor.
   Test A shows that pinning $W^2_{qk}$ at zero freezes layer 1 permanently.

3. **Layer 2 is the one with signal, and it opens the gate.** Its gradient has no $W^2$ prefactor, and the
   position one-hots make the position-block of $\partial L/\partial W^2$ equal to $\partial L/\partial S^2$
   entry for entry, so it cannot vanish by cancellation. One step later $W^2\neq0$, and layer 1 starts.

4. **The dependence is exactly proportional.** $\|\partial L/\partial W^1_{qk}\| \propto \|W^2_{qk}\|$ with a
   fitted log–log slope of ~1 across four decades (section 3). Layer 2 is a valve, not a switch: early on
   $W^2$ is small, so only a trickle reaches layer 1 — which is why layer 1 stays far behind for hundreds
   of steps even though it is unblocked from step 2.

5. **Relation to the random-init claim.** `meeting_notes.txt` records that at random init layer 2 already
   has the useful gradient signal while layer 1 has none, i.e. *the second layer forms first*. Zero init
   turns that empirical asymmetry into an exact statement with a proof, and identifies the mechanism:
   layer 1's learning signal is **routed through** $W^2_{qk}$.

6. **Asymmetry in the strength of the two claims.** Layer 1's vanishing is an identity (multiplication by
   zero) — always true, any batch, any dataset. Layer 2's non-vanishing is generic — it would take exact
   cancellation across all 256 sequences in every one of $T^2$ entries. Worth one sentence in the write-up
   so a reviewer does not ask.

7. **The first gradient already knows where to go.** Two independent measurements say so, both taken
   at step 1 from a matrix with no structure to inherit:
   * **46.5%** of the layer-2 gradient's energy lands in the construction's support, which is 0.41% of
     the matrix — a **113x** enrichment. That is the zero-init analogue of the 0.48-0.89 figure recorded
     at random init in `zero_init_analysis.ipynb` §4.
   * Inside the induction block, GD pushes the **diagonal 12.2x harder** than the off-diagonal
     ($+7.26\times10^{-5}$ vs $+5.93\times10^{-6}$).

   So "layer 2 forms first" is not only about *when* it moves — the very first gradient it receives is
   already aimed at the induction circuit.

8. **Why the epoch-resolution notebook cannot see any of this.** One epoch is 98 steps and `snap_every=1`,
   so snapshot row 1 is already step 98 — five times past the end of this notebook's window. The freeze
   happens and is long over before the first post-init snapshot is written, which is exactly why this
   notebook exists.

Figures and GIFs are written to `artifacts/figures_zero_step/`.